In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r"C:\Users\HP\Desktop\Projects - Data Analysis\RSF\RSF\imputed_ckm_data.csv")
df

CKM Staging into Advanced and Non-advanced

In [ ]:
def binary_ckm(row):
    ckm = row['ckm_stage']
    if ckm > 2:
        return 1
    if ckm <= 2:
        return 0

binary_col = df.apply(binary_ckm, axis=1)
binary_df = pd.concat([df, pd.Series(binary_col, name= 'binary_ckm')], axis=1)

binary_df

Variation Inflation Factor

In [ ]:
df_edited = binary_df.copy()
df_edited = binary_df.fillna(999)
df_binary = df_edited.drop(columns=['ckm_stage'])

In [ ]:
df_binary.columns

In [ ]:
X = df_binary.copy()
vif_data = pd.DataFrame()
vif_data['features'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

vif_data

In [ ]:
remove_cols = list()
rem_data = vif_data.loc[vif_data['VIF'] >= 10.00, 'features']
for col in rem_data.values:
    remove_cols.append(col)

vif_removed_df = df_binary.drop(columns=remove_cols)

In [ ]:
vif_removed_df.info()


Binary Logistic Regression

In [ ]:
import statsmodels.formula.api as smf
sent = "nationality"
for col in vif_removed_df.columns:
    if (col != 'binary_ckm' or col != 'encounter_date') and not pd.api.types.is_string_dtype(vif_removed_df[col]):
        sent += f"+ {col} "

formula = (f'binary_ckm ~ {sent}')
ckm_model = smf.logit(formula = formula, data=binary_df).fit()
print(ckm_model.summary())